# 7 · Temporal SSM: a Phase per Cycle

*Phasor networks, from the ground up — notebook 7 of 7.*

So far time played a supporting role: drive an oscillator with a *constant* input
and its phase **settles** — a temporal series converging to a single atemporal
value (the static answer of notebooks 2–6). The **state-space (SSM)** view turns
time into a first-class carrier: a layer maps an input phase *sequence*
`(channels, L cycles, batch)` to an output phase **per cycle**.

The engine is the discrete kernel `K[n] = Aⁿ·B` with `A = exp(k·Δt)`,
`B = (A−1)/k` — the exact unrolling of `dz/dt = k·z + I(t)` (`phasor_kernel`,
`causal_conv_dirac` in `kernels.jl`). When the input is constant, each cycle
reproduces the settled value; when the input varies per cycle, each cycle carries
its own phase.

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
using PhasorNetworks
using Plots, Lux
using Random: Xoshiro

## The kernel sets the memory

Each channel's impulse response `K[n] = Aⁿ·B` decays at rate `λ`. Slow channels remember many cycles; fast ones forget quickly — multi-timescale memory.

In [ ]:
L_k = 200; dt = 0.01f0
K = phasor_kernel(Float32[-0.1, -0.3, -1.0], Float32[2pi, 2pi, 2pi], dt, L_k)
plot(abs.(K)', xlabel="lag n", ylabel="|K[n]|", label=["λ=-0.1" "λ=-0.3" "λ=-1.0"],
     title="impulse responses (memory timescales)")

## A constant input settles — the atemporal limit

Feed the same phase every cycle. `PhasorDense` on a 3-D Phase input `(C, L, B)` returns a phase per cycle; with constant drive each cycle reproduces the settled value, so the per-cycle output is flat. This is the convergence-to-a-limit regime.

In [ ]:
rng = Xoshiro(1)
C = 6; B = 1; L = 24
layer = PhasorDense(C => C, normalize_to_unit_circle, init_mode=:hippo, use_bias=false)
ps, st = Lux.setup(rng, layer)

x_const2d = random_symbols(rng, (C, B))
x_const = repeat(reshape(x_const2d, (C, 1, B)), 1, L, 1)   # constant across L cycles
y_const, _ = layer(x_const, ps, st)                         # (C, L, B)

plot(Float32.(y_const[:, :, 1])', xlabel="cycle", ylabel="output phase", legend=false,
     title="constant input → settled per-cycle phase")

## A time-varying input carries a sequence

Now let the input phase change every cycle (a slow ramp per channel). The per-cycle output *tracks* it — a genuine temporal phase sequence, not a convergence.

In [ ]:
wrap(t) = Float32(mod(t + 1, 2) - 1)         # keep phases in [-1, 1)
ramp = Float32.(range(-1, 1, L))
x_var = stack([Phase.(wrap.(ramp .+ 0.3f0 * (c - 1))) for c in 1:C], dims=1)  # (C, L)
x_var = reshape(x_var, (C, L, B))
y_var, _ = layer(x_var, ps, st)

p_in = plot(Float32.(x_var[:, :, 1])', xlabel="cycle", ylabel="input phase", legend=false, title="input: per-cycle phases")
p_out = plot(Float32.(y_var[:, :, 1])', xlabel="cycle", ylabel="output phase", legend=false, title="output: per-cycle phases")
plot(p_in, p_out, layout=(1, 2), size=(820, 320))

## The spiking side

The same per-cycle picture exists in continuous time:

- `ssm_phases_to_train(phases)` encodes a 3-D phase sequence as spikes with
  **timestep *l* → oscillation period *l*** (one set of spikes per cycle), unlike
  `phase_to_train` which repeats one phase every cycle;
- `sample_phases_at_periods(sol, L, spk_args)` reads a phase back out at each period
  boundary of an ODE solution.

Together they let the oscillator ODE compute the very same per-cycle phase sequence
the discrete kernel produces (see `ssm_equivalence` / `phasor_ssm`).

## Series wrap-up

| # | notebook | idea |
|---|----------|------|
| 1 | phase triple | one phase, three faces |
| 2 | oscillators | a phase held in time |
| 3 | operations | similarity / bundling / binding by superposition & rotation |
| 4 | resonator | factor bound products by iterated cleanup |
| 5 | graph | store & query structure in one vector |
| 6 | neural nets | train phasor layers by gradient descent |
| 7 | temporal SSM | a phase **per cycle** — time as the carrier |

From a single angle on the unit circle to learned, time-unrolled computation —
all the same phase algebra, run atemporally or on oscillating neurons.